# Unsolicited Message Classifier
This is a simplified script to demonstrate key features of the classification task. This script reads a single row of a synthetic dataset and outputs classifications for that row. 

## Setup

In [ ]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

This demo is configured for OpenAI chat completions API, it will plug and play with an OpenAI API key but will have to be reconfigured for use with other providers

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-03"

## Structured Output Schema
Structured outputs allow us to enforce a JSON schema to generated outputs, ideal for this kind of systematic, repeated classification task.

In [ ]:
class ThemeItem(BaseModel):
    key: str = Field(description="Theme identifier key")
    score: float = Field(description="Confidence score between 0.0 to 1.0", ge=0.0, le=1.0)
    present: bool = Field(description="True if theme is detected (score >= 0.5)")
    evidence: str = Field(description="Brief supporting reasoning for classification.")

class ClassificationResponse(BaseModel):
    themes: list[ThemeItem]

## System Prompt and Theme Definitions

In [ ]:
system_prompt = """
**Dataset description:** These data come from an automated text messaging program that sends daily dialogues to help the user self-manage mental health symptoms. Usually, users write back to specific prompts or questions from the system, but the users are also able to send other free text messages to the system whenever they want. 
**Study goal and research questions:** Here, we are trying to better understand the different types or categories of free text messages users send that are unsolicited (that means, they’re not in response to a system prompt. You will help us decide whether specific unsolicited user messages fall in some of the major categories we have identified.

You will be given:  
1. A block labelled "Context (messages for the day):" that shows prior messages exchanged.  
2. A single line labelled "Target message:" that contains the user message we want to classify. 
3. A line labeled "previous_system_message" containing the system message immediately preceding the Target message.  
  
**Task**  
For the Target message only, decide whether each of the themes below is PRESENT (True) or NOT PRESENT (False).

**Assumptions and constraints**
• A theme is PRESENT when its confidence score ≥ 0.50.
• At least one theme must be marked PRESENT in every output.
- If none of the themes reaches the 0.50 threshold, select the theme with the highest confidence score, set its present field to true, and (optionally) raise its score to 0.50 to preserve internal consistency.
• Evidence must briefly justify each label using information from the Target message only.
• Produce only the JSON array—no additional commentary.  

**Output format**  
Return a JSON array of exactly NINE objects, one per theme, in the order shown above, following this schema:
class ThemeItem(BaseModel):  
    key: str                 # Theme identifier  
    evidence: str            # Description of your reasoning for classification with evidence from the text.
    score: float             # Confidence 0.0-1.0  
    present: bool            # True if score ≥ 0.5 else False  
  
**Example structure (values are illustrative):**  
  
[  
  {"key":"polite","evidence":"Message is substantive, not mere pleasantry","score":0.05,"present":false},  
  {"key":"relational","evidence":"No overt emotional connection to system","score":0.15,"present":false}, 
  {"key":"provocative","evidence":"No signs of provocation","score":0.10,"present":false},  
  {"key":"journaling","evidence":"User reflects on coping techniques they studied","score":0.82,"present":true},  
  {"key":"mistimed","score":0.05,"present":false,"evidence":"Not an answer to earlier prompt"},   
  {"key":"commands","score":0.05,"present":false,"evidence":"No request to control the program"},  
  {"key":"reaction","score":0.01,"present":false,"evidence":"Not an SMS reaction format"},  
  {"key":"automated","score":0.01,"present":false,"evidence":"No auto-reply language"},  
  {"key":"na","score":0.05,"present":false,"evidence":"Message is coherent"}  
]  
"""

theme_list = {
    "polite": 
        """
        the whole message is a perfunctory/polite social response with little substantive intent (e.g. “okay”, “good morning”).
        """,
    "relational": 
        """  
        Definition
        Label a message “relational” when the user’s primary goal is to create an emotional or personal connection with the system. This is true when EITHER of the following holds:
        a. The message contains explicit emotional content (affectionate wording, personifying the bot, multiple “!” or emojis conveying feeling).
        b. The message is very short (≈1–3 tokens) and includes expressive punctuation or an emoji (e.g., “Hello!”, “Thanks!”, “👍”). Empirically, these brief, emotive messages almost always function as social pleasantries, so default to relational = True unless decisive evidence suggests otherwise.

        Decision Rules

        Emotional signal required: The text must meet criterion (a) or (b) above.
        More than routine politeness: Messages that are polite but lack emotional markers (“Hi,” “Thanks.”) remain only “polite,” not “relational.”
        Can coexist with “polite”: If a message meets rule 1 and is also courteous, tag both “polite” and “relational.”

        Examples
        • “Thanks so much!!! 😊 You’re the best.” → relational = True (polite + strong emotional cues)
        • “Good morning, friend bot! Hope you’re having a great day ❤️” → relational = True (greeting + affectionate language)
        • “Hi” → relational = False (polite only; no emotional markers)
        • “Okay.” → relational = False (perfunctory acknowledgement)
        • “👍” → relational = True (single emoji reaction)
        """,
    "provocative": 
        """
        Definition
        Tag a message “provocative” when the user is pushing back on, challenging, or expressing frustration with the program or its content. OR if a user is asking for clarity/elaboration on program content. This includes, but is not limited to:

        a. Inflammatory or irrelevant remarks intended to provoke (“This app is useless,” “Are you even listening?”). This includes cries for help that are out of context. 
        b. Sarcasm, ridicule, or absurd statements aimed at the system (“Sure, that’ll fix my life 🙄”).
        c. Requests for explanation or clarification (“What does that exercise do?”, “Explain how this helps anxiety”).
        d. Direct disagreement or uncooperativeness (“That doesn’t make sense,” “I already told you I’m not anxious!”).

        Edge Cases
        • Polite wording does not negate provocation if the message questions or contradicts the system.
        • Multiple themes may apply; always include “provocative” when criteria above are met.
        """,
    "journaling": 
        """
        Definition
        Tag a message “journaling” when the user is recording, reflecting on, or planning their own thoughts, feelings, or actions—essentially writing a “note to self.” The entry can be long and narrative or very brief (1-2 clauses). Even terse statements such as “Yes, it is hard” or “Okay, I will” qualify if they express a personal reaction or intention.

        Decision Rules

        Self-focused content: The text must describe the user’s internal state, experience, or intention (past, present, or future).
        Not primarily addressive: Messages aimed mainly at the bot (greetings, thanks, questions, or commands) are not journaling unless they also contain self-reflection.
        Length-independent: Both detailed reflections and short affirmations/commitments count.
        May coexist with other themes: A message like “Thanks, I’m feeling better today” can be both polite and journaling.
        Exclusions:
        • Exclude common interaction/command keywords like "Yes", "no", "A", "B", "C", "D", "1", "2", "3", "4", "Low", "Medium", "High", etc.
        • Exclude pure greetings (“Hi,” “Good morning!”) without self-content.
        • Exlcude requests or commands directed at the system (“Send that again, please”).
        • Exclude solely informational data (“3/10 on mood scale”) unless accompanied by reflection (“3/10 today—really struggling”).
        Examples
        • “I feel anxious but hopeful after yesterday’s exercise.” → journaling = True (self-reflection)
        • “Yes, it is hard.” → journaling = True (brief acknowledgment of personal difficulty)
        • “Okay, I will try that tonight.” → journaling = True (commitment / future plan)
        • “Stayed up all night again.” → journaling = True (record of personal experience)
        • “Can you explain the next step?” → journaling = False (question to system)
        • “Hi!” → journaling = False (greeting only)

        Edge Cases
        • If a message contains both a personal reflection and a request (“I’m really stressed—what should I do?”), mark both journaling (self-reflection) and commands or provocative as appropriate.
        • Single-word reactions like “Hard.” or “Exhausted.” count as journaling because they log internal state.
        """,
    "mistimed": 
        """
        ## Key Definitions & Assumptions

        1. **Response-elicitation prompt (REP)**
        • A system message qualifies as a REP only if it contains at least one handwriting emoji (✍).
        • A question without ✍ is **not** a REP.

        2. **mistimed** (theme key: `"mistimed"`)
        The Target is mistimed when it is an obvious, belated answer to a prior REP that is no longer the active prompt.

        ---

        ## Decision Rules for `"mistimed"`

        1. **Direct answer required** – The Target must clearly supply the information requested by the REP (rating, yes/no, name, etc.).
        2. **Prompt no longer current** – At least one intervening non-prompt message (thank-you, link, reminder, etc.) appears between the REP and the Target.
        3. **No ambiguity** – If the Target can reasonably be read as anything other than a delayed reply, mark `mistimed = false`.

        ---

        ## Edge Cases

        `mistimed = false`
        • Target introduces new content, opinions, or emotions unrelated to a REP.
        • Ambiguous timing—unclear if the Target answers a past REP.

        `mistimed = true`
        • Technical issues (e.g., SMS or logging errors) place a user reply **before** its corresponding REP. If the content plainly answers that later REP, still mark `mistimed = true`.

        ---

        ## Examples

        1.
        Context:
        to-recipient: ✍ How helpful were these messages? 1-5 ✍
        to-recipient: Thank you for your feedback!
        to-system: 3

        Target → 3

        Result → present = true (belated rating; REP no longer current)

        2.
        Context:
        to-recipient: ✍ What value matters most to you? ✍
        to-recipient: Thank you for your response!
        to-system: My nephews.

        Target → My nephews.

        Result → present = true (late answer)

        3.
        Context:
        to-recipient: ✍ What value matters most to you? ✍
        to-system: My nephews.
        to-recipient: Thank you for your response!

        Target → My nephews.

        Result → present = false (on-time answer)

        4.
        Context:
        to-system: Maybe go to the mall with my mom.
        to-recipient: ✍ What might be a fun activity to do with someone you care about? ✍
        to-recipient: Thank you for sharing!

        Target → Maybe go to the mall with my mom.

        Result → present = true (reply logged before REP)

        5.
        Context:
        to-recipient: ✍ What might be a fun activity to do with someone you care about? ✍
        to-system: Love going dancing
        to-recipient: Thank you for sharing!
        to-system: At the club or even dancing alone

        Target → At the club or even dancing alone

        Result → present = false (follow-up, not mistimed)

        6.
        Context:
        to-recipient : Good morning! Today we'll spend more time on social connectedness. When you have more positive contact with others, you are more likely to have positive thoughts about yourself and your life.
        to-recipient : Making plans can help us follow through when we need more social contact. Read more: https://mhatxt.co/u/t7xj26pmo39winzc
        to-system : Okay
        to-recipient : Is there someone who tends to brighten your day? Today, you might reach out to see if they'd like to have coffee or some other kind of get together.
        to-system : ​❤️​ to “ Is there someone who tends to brighten your day? Today, you might reach out to see if they'd like to have coffee or some other kind of get together. ”
        to-recipient : ✍ What might be a fun activity to do with someone you care about? ✍

        Target → ❤️​ to “ Is there someone who tends to brighten your day? Today, you might reach out to see if they'd like to have coffee or some other kind of get together. ”

        Result → present = false (Target message not relevant for REP)

        7.
        Context:
        to-recipient : Good morning! Today we'll send more messages about relaxation. This is all about building simple skills to find a sense of calm when you need it.
        to-system : Okay
        to-recipient : One technique for physical relaxation is called progressive muscle relaxation: https://mhatxt.co/u/c18oweqpp1en980q

        Target → Okay

        Result → present = false (not relevant for any REP)

        ---

        ## Notes

        • Verify the prompt is a REP (contains ✍).
        • The JSON array must be the sole output.
        • score and present must be consistent (present = score ≥ 0.5). 
        """,
    "commands": 
        """
        inquiries about how the program works or attempts to control it (e.g. “pause”, “skip”, “unsubscribe”) or feedback about the system. 
        """,
    "reaction": 
        """
        Definition
        Label “reaction” only when the incoming text is an auto-generated SMS-style reaction to a previous message, typically formatted like:

        • “Reacted 👍 to ‘Thanks for checking in.’”
        • “Reacted ❤️ to ‘Here’s today’s prompt.’”

        Key Properties
        • Machine-generated template that cites the original message in quotation marks.
        • Contains no new user-authored content beyond the emoji reaction.

        Exclusivity Rule
        If a message is tagged reaction = True, DO NOT assign any other theme in the same output row. The absence of substantive user text makes every other category inapplicable.

        Decision Rules

        Template match: Message follows the “Reacted to ‘’” pattern (minor wording or punctuation variants acceptable).
        No additional text: Any extra words from the user (“Thanks! ❤️”) disqualify it; evaluate other themes instead.
        Exclusivity enforcement: When rule 1 is met, set present = True for reaction, present = False for all eight other themes, and assign low confidence scores (≈0.01) to those eight.
        Examples
        • “Reacted 😂 to ‘Great job on today’s exercise!’” → reaction = True; all other themes = False
        • “👍” → reaction = False (stand-alone emoji; evaluate for relational, journaling, etc.)
        • “Reacted ❤️ Thanks!” → reaction = False (contains extra text; treat as normal message)
        """,
    "automated": 
        """
        auto-reply texts generated by phone focus/drive/work modes indicating unavailability. 
        """,
    "na": 
        """
        message is uninterpretable or clearly miscoded data.
        """,
    }

## Load Data

In [ ]:
df = pd.read_excel("../data/synthetic_input_data.xlsx")
df.head()

In [ ]:
# Change this index to try a different row
ROW_INDEX = 0

row = df.iloc[ROW_INDEX]
print(f"Target message: {row['Message']}")

## Classify

In [ ]:
def classify_message(context: str, message: str) -> ClassificationResponse:
    user_content = (
        f"Context (messages for the day):\n{context}\n\n"
        f"Target message:\n{message}"
    )
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt + "\n\n" + theme_list},
            {"role": "user",   "content": user_content}
        ],
        response_format=ClassificationResponse
    )
    return response.choices[0].message.parsed

In [ ]:
result = classify_message(
    context=row["days_transcript"],
    message=row["Message"]
)

print(json.dumps([theme.model_dump() for theme in result.themes], indent=2))